## Intertemporal decomposition notebook

- Run this after you've ran the parse_experiment_in_individual_files.py
- After this you can run CB

In [1]:
import os
import pandas as pd
import numpy as np
from utils import decomposition

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
td = decomposition.TemporalDecomposition()

In [4]:
# Set up paths
SCRIPT_DIR_PATH = os.getcwd()
CW_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "cw")
DATA_DIR_PATH = os.path.join(SCRIPT_DIR_PATH, "data")
ENSEMBLE_DATA_DIR_PATH = os.path.join(DATA_DIR_PATH, "ensemble_data")

In [5]:
# Load emissions targets
te_all = pd.read_csv(os.path.join(CW_DIR_PATH, "emission_targets_louisiana.csv"))
target_country = "LA"
cols_needed = ["Subsector", "Gas", "Vars", "Edgar_Class", target_country]
te_all = te_all[cols_needed].copy()
te_all["tvalue"] = te_all[target_country]
te_all = te_all.drop(columns=[target_country])
te_all

,Subsector,Gas,Vars,Edgar_Class,tvalue
0,lvst,ch4,emission_co2e_ch4_lvst_entferm_buffalo:emissio...,AG - Livestock:CH4,1.539811
1,lsmm,ch4,emission_co2e_ch4_lsmm_anaerobic_digester:emis...,AG - Livestock:CH4,0.151107
2,lsmm,n2o,emission_co2e_n2o_lsmm_direct_anaerobic_digest...,AG - Livestock:N2O,0.079212
3,agrc,co2,emission_co2e_co2_agrc_biomass_bevs_and_spices...,AG - Crops:CO2,0.000000
4,agrc,ch4,emission_co2e_ch4_agrc_anaerobicdom_rice:emiss...,AG - Crops:CH4,2.384974
...,...,...,...,...,...
63,soil,co2,emission_co2e_co2_soil_lime_use:emission_co2e_...,LULUCF - Organic Soil:CO2,0.305152
64,soil,n2o,emission_co2e_n2o_soil_fertilizer:emission_co2...,LULUCF - Organic Soil:N2O,0.928029
65,ccsq,ch4,emission_co2e_ch4_ccsq_direct_air_capture,CCSQ:CH4,0.000000
66,ccsq,co2,emission_co2e_co2_ccsq_direct_air_capture,CCSQ:CO2,0.000000


In [6]:
# Parse target variables
te_all["Vars_list"] = te_all["Vars"].str.split(":")
target_vars = [item for sublist in te_all["Vars_list"].tolist() for item in sublist]
print("Target variables:", target_vars[:10])  # Display first 10 target variables
print("Total target variables:", len(target_vars))

Target variables: ['emission_co2e_ch4_lvst_entferm_buffalo', 'emission_co2e_ch4_lvst_entferm_cattle_dairy', 'emission_co2e_ch4_lvst_entferm_cattle_nondairy', 'emission_co2e_ch4_lvst_entferm_chickens', 'emission_co2e_ch4_lvst_entferm_goats', 'emission_co2e_ch4_lvst_entferm_horses', 'emission_co2e_ch4_lvst_entferm_mules', 'emission_co2e_ch4_lvst_entferm_pigs', 'emission_co2e_ch4_lvst_entferm_sheep', 'emission_co2e_ch4_lsmm_anaerobic_digester']
Total target variables: 470


In [7]:
# Output folders
ensemble_id = "2025-08-17T22;36;58.136929"
RUN_ENSEMBLE_DIR_PATH = os.path.join(ENSEMBLE_DATA_DIR_PATH, f"sisepuede_summary_results_run_sisepuede_run_{ensemble_id}")
PARSED_RUNS_DIR_PATH = os.path.join(DATA_DIR_PATH, "parsed_runs")
ENSEMBLE_PARSED_DIR_PATH = os.path.join(PARSED_RUNS_DIR_PATH, ensemble_id)
files_names = [f for f in os.listdir(ENSEMBLE_PARSED_DIR_PATH) if f.endswith('.csv')]

In [11]:
print(f"Number of parsed run files found: {len(files_names)}")

Number of parsed run files found: 12


In [12]:
PARSED_RUNS_PROCESSED_DIR_PATH = os.path.join(DATA_DIR_PATH, "rescaled_parsed_runs")
ENSEMBLE_PARSED_PROCESSED_DIR_PATH = os.path.join(PARSED_RUNS_PROCESSED_DIR_PATH, ensemble_id)
os.makedirs(PARSED_RUNS_PROCESSED_DIR_PATH, exist_ok=True)
os.makedirs(ENSEMBLE_PARSED_PROCESSED_DIR_PATH, exist_ok=True)

In [13]:
time_period_ref = 7

# --- process each batch file with your updated rescale() ---
for run, output_file in enumerate(files_names):
    path_in = os.path.join(ENSEMBLE_PARSED_DIR_PATH, output_file)
    df_in = pd.read_csv(path_in)

    # keep only years >= t0
    df_in = df_in[df_in["time_period"] >= time_period_ref].copy()
    if df_in.empty:
        print(f"[skip] {output_file} has no rows >= t0")
        continue

    # single region assumption (but still grab it from data)
    region = df_in["region"].iloc[0]

    # choose a baseline id that exists in THIS batch
    ref_id = 0 if (df_in["primary_id"] == 0).any() else int(df_in["primary_id"].min())

    # run rescale; rescale writes its own output file
    td.rescale(
        z=0,
        rall=np.array([region]),
        data_all=df_in,
        te_all=te_all,
        initial_conditions_id=[ref_id],
        dir_output=ENSEMBLE_PARSED_PROCESSED_DIR_PATH,
        time_period_ref=time_period_ref,
        run=run
    )
    print(f"[done] run={run} file={output_file} region={region} baseline_id={ref_id}")


Saved: /Users/tony/Documents/sisepuede_modeling/ssp_louisiana/1000_runs_ensamble_postprocessing/data/rescaled_parsed_runs/2025-08-17T22;36;58.136929/louisiana_0.csv
[done] run=0 file=6.csv region=louisiana baseline_id=74074
Saved: /Users/tony/Documents/sisepuede_modeling/ssp_louisiana/1000_runs_ensamble_postprocessing/data/rescaled_parsed_runs/2025-08-17T22;36;58.136929/louisiana_1.csv
[done] run=1 file=7.csv region=louisiana baseline_id=75075
Saved: /Users/tony/Documents/sisepuede_modeling/ssp_louisiana/1000_runs_ensamble_postprocessing/data/rescaled_parsed_runs/2025-08-17T22;36;58.136929/louisiana_2.csv
[done] run=2 file=5.csv region=louisiana baseline_id=73073
Saved: /Users/tony/Documents/sisepuede_modeling/ssp_louisiana/1000_runs_ensamble_postprocessing/data/rescaled_parsed_runs/2025-08-17T22;36;58.136929/louisiana_3.csv
[done] run=3 file=4.csv region=louisiana baseline_id=72072
Saved: /Users/tony/Documents/sisepuede_modeling/ssp_louisiana/1000_runs_ensamble_postprocessing/data/res

In [14]:
# collect decomposed runs
files_out = [f for f in os.listdir(ENSEMBLE_PARSED_PROCESSED_DIR_PATH) if f.endswith(".csv")]
data_complete = pd.concat(
    [pd.read_csv(os.path.join(ENSEMBLE_PARSED_PROCESSED_DIR_PATH, f)) for f in files_out],
    ignore_index=True
)

# (optional) dedupe exact duplicates
key_cols = ["region", "primary_id", "time_period"]
data_complete = data_complete.drop_duplicates(subset=key_cols + [c for c in data_complete.columns if c not in key_cols])

In [15]:
data_complete.head()

,Index,time_period,primary_id,region,area_agrc_crops_bevs_and_spices,area_agrc_crops_cereals,area_agrc_crops_fibers,area_agrc_crops_fruits,area_agrc_crops_herbs_and_other_perennial_crops,area_agrc_crops_nuts,...,yf_agrc_herbs_and_other_perennial_crops_tonne_ha,yf_agrc_nuts_tonne_ha,yf_agrc_other_annual_tonne_ha,yf_agrc_other_woody_perennial_tonne_ha,yf_agrc_pulses_tonne_ha,yf_agrc_rice_tonne_ha,yf_agrc_sugar_cane_tonne_ha,yf_agrc_tubers_tonne_ha,yf_agrc_vegetables_and_vines_tonne_ha,yf_lndu_supremum_pastures_tonne_per_ha
0,louisiana_71071,7,71071,louisiana,0.0,356696.043492,66146.621297,77.773211,76769.151304,6508.599365,...,12.023165,2.949341,6.177415,0.0,3.474771,8.253027,87.719298,39.935028,30.906144,92.81
1,louisiana_71071,8,71071,louisiana,0.0,355221.860914,65873.245131,77.451784,76451.873477,6481.700093,...,12.023165,2.949341,5.189029,0.0,2.688411,8.475414,83.271559,39.935028,30.906144,92.81
2,louisiana_71071,9,71071,louisiana,0.0,353750.075712,65600.313541,77.130879,76135.111621,6454.844566,...,12.023165,2.949341,6.319971,0.0,3.416092,8.422668,89.334930,39.935028,30.906144,92.81
3,louisiana_71071,10,71071,louisiana,0.0,352280.764654,65327.840762,76.810514,75818.882257,6428.034184,...,12.023165,2.949341,6.319971,0.0,3.416092,8.422668,89.334930,39.935028,30.906144,92.81
4,louisiana_71071,11,71071,louisiana,0.0,350814.002751,65055.840705,76.490704,75503.201530,6401.270317,...,12.023165,2.949341,6.319971,0.0,3.416092,8.422668,89.334930,39.935028,30.906144,92.81


In [16]:
time_period_ref

7

In [17]:
# --- GLOBAL t0 EQUALIZATION (Option A) ---
# mapped_only=False => equalize all co2e_ vars; set True to equalize only those in te_all.Vars_list
data_complete = td.enforce_global_t0_equalization(
    data_complete,
    time_period_ref=time_period_ref,
    te_all=te_all,
    mapped_only=False
)

data_complete = td.recompute_subsector_totals(data_complete, te_all)

# --- VALIDATE ---
print(td.assert_equal_t0(data_complete, 7))           # base vars
print(td.assert_equal_t0_totals(data_complete, 7))    # subsector totals


True
True


In [18]:
# # Filter some outlier runds
# # primary_ids_to_remove = [354920, 355090, 354624]
# primary_ids_to_remove = [354790, 354833, 355220]
# data_complete = data_complete[~data_complete["primary_id"].isin(primary_ids_to_remove)]
# data_complete.shape

In [19]:
# check for fields full of nans
data_complete.isnull().sum()

Index                                     0
time_period                               0
primary_id                                0
region                                    0
area_agrc_crops_bevs_and_spices           0
                                         ..
yf_agrc_rice_tonne_ha                     0
yf_agrc_sugar_cane_tonne_ha               0
yf_agrc_tubers_tonne_ha                   0
yf_agrc_vegetables_and_vines_tonne_ha     0
yf_lndu_supremum_pastures_tonne_per_ha    0
Length: 3979, dtype: int64

In [20]:
data_complete.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 348 entries, 0 to 347
Columns: 3979 entries, Index to yf_lndu_supremum_pastures_tonne_per_ha
dtypes: float64(3965), int64(12), object(2)
memory usage: 10.6+ MB


In [21]:
# --- WRITE FINAL OUTPUT ---
final_name = f"sisepuede_results_IDE_{ensemble_id}.csv"
out_path = os.path.join(RUN_ENSEMBLE_DIR_PATH, final_name)
data_complete.to_csv(out_path, index=False)
print(f"[final] wrote {out_path}")

[final] wrote /Users/tony/Documents/sisepuede_modeling/ssp_louisiana/1000_runs_ensamble_postprocessing/data/ensemble_data/sisepuede_summary_results_run_sisepuede_run_2025-08-17T22;36;58.136929/sisepuede_results_IDE_2025-08-17T22;36;58.136929.csv
